In [1]:
#******************************************************
#*
#* Name:         nb-03-pyspark-ingest-csv-files
#*     
#* Design Phase:
#*     Author:   John Miner
#*     Date:     04-01-2026
#*     Purpose:  Load delta table using spark.
#* 
#******************************************************/

StatementMeta(, 64c7b607-f5fb-49fa-a257-60c3f68ef2a0, 3, Finished, Available, Finished, False)

In [2]:
#
#  1 - drop existing table
#

df = spark.sql("DROP TABLE IF EXISTS bronze.spark_stock_data")
display(df)

StatementMeta(, 64c7b607-f5fb-49fa-a257-60c3f68ef2a0, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2dabaf76-9203-4a27-9816-05dca5451f6c)

In [3]:
#
#  2 -  use spark.read since it has the most options (standard session - 38s)
#

from pyspark.sql.functions import *

# define path
path = "Files/stocks"

# define schema
custom_schema = """
    symbol string, 
    date string,
    open float,
    high float,
    low float,
    close float,
    adjclose float,
    volume bigint
"""

# read in csv data
df1 = (
  spark.read.format("csv")                    
  .option("sep", ",")        
  .option("header", "true")
  .option("inferSchema", "false")  
  .option("recursiveFileLookup", "true")
  .schema(custom_schema)
  .load(path)               
)

# get rid of file names
df2 = df1.filter(~col("symbol").contains(".CSV")).filter(col("volume") != lit(0))

# add prefix
df2 = df2.select([col(c).alias("st_" + c) for c in df2.columns])

# write delta table
df2.write.saveAsTable("bronze.spark_stock_data")

StatementMeta(, 64c7b607-f5fb-49fa-a257-60c3f68ef2a0, 5, Finished, Available, Finished, False)

In [4]:
#
#  3 - how many records ingested?
#

df = spark.sql("SELECT COUNT(*) AS Total FROM bronze.spark_stock_data")
display(df)

StatementMeta(, 64c7b607-f5fb-49fa-a257-60c3f68ef2a0, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bca38a38-f590-4769-a861-11b9d8f86271)

In [5]:
%%sql

--
-- 4 - recs by year for AMZN stock
--

select 
  st_symbol, 
  substring(st_date, 7, 4) as st_year, 
  count(*) as st_total
from bronze.spark_stock_data where st_symbol = 'AMZN'
group by st_symbol, substring(st_date, 7, 4)

StatementMeta(, 64c7b607-f5fb-49fa-a257-60c3f68ef2a0, 7, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 3 fields>

In [6]:
#
#  5 - drop existing table
#

df = spark.sql("DROP TABLE IF EXISTS bronze.spark_one_stock_file")
display(df)

StatementMeta(, 64c7b607-f5fb-49fa-a257-60c3f68ef2a0, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 10a1de0e-5884-40fa-9b0c-5df06526f359)

In [7]:
#
#  6 - use spark.read since it has the most options (standard session - 20s)
#

from pyspark.sql.functions import *

# define path
path = "Files/SNP-5YEARS.CSV"

# define schema
custom_schema = """
    symbol string, 
    date string,
    open float,
    high float,
    low float,
    close float,
    adjclose float,
    volume bigint
"""

# read in csv data
df3 = (
  spark.read.format("csv")                    
  .option("sep", "\t")        
  .option("header", "true")
  .option("inferSchema", "false")  
  .option("recursiveFileLookup", "true")
  .schema(custom_schema)
  .load(path)               
)

# get rid of file names
df4 = df3.filter(~col("symbol").contains(".CSV")).filter(col("volume") != lit(0))

# add prefix
df4 = df4.select([col(c).alias("st_" + c) for c in df4.columns])

# write delta table
df4.write.saveAsTable("bronze.spark_one_stock_file")

StatementMeta(, 64c7b607-f5fb-49fa-a257-60c3f68ef2a0, 9, Finished, Available, Finished, False)

In [8]:
%%sql

--
--  7 - how many records ingested?
--

select count(*) as total
from bronze.spark_one_stock_file


StatementMeta(, 64c7b607-f5fb-49fa-a257-60c3f68ef2a0, 10, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [9]:
%%sql

--
-- 8 - recs by year for MSFT stock
--

select 
  st_symbol, 
  substring(st_date, 7, 4) as st_year, 
  count(*) as st_total
from bronze.spark_one_stock_file where st_symbol = 'MSFT'
group by st_symbol, substring(st_date, 7, 4)


StatementMeta(, 64c7b607-f5fb-49fa-a257-60c3f68ef2a0, 11, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 3 fields>

In [10]:
%%sql

select * from bronze.spark_one_stock_file where st_volume = 0

StatementMeta(, 64c7b607-f5fb-49fa-a257-60c3f68ef2a0, 12, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 8 fields>